# 02 — Explanations: Grad-CAM and Integrated Gradients

**Project:** Quantitatively Audited Explainable AI for Brain Tumor MRI Classification

This notebook **generates** explanations for the baseline ResNet18 from notebook 1. It doesn't
judge them; that's notebook 3 (faithfulness, stability, localization). What it does:

1. loads the trained checkpoint and confirms it reproduces notebook 1's test predictions
2. implements Grad-CAM and Integrated Gradients behind one interface in `src/explain/`
3. computes both maps for **every test image** and saves them, together with the resized tumor
   masks, keyed to the row order of `test_predictions.csv`
4. shows a curated, seeded set of correct and misclassified cases

**Inputs to attach** (Add Input → Your Work → Notebooks):

| input | used for | on `sys.path`? |
|---|---|---|
| notebook 1 output | checkpoint, `test_predictions.csv`, `test_metrics.json`, `src/` | **yes, its `src/` only** (copied, see §2b) |
| notebook 0 output | `manifest.csv` + the `.mat` files | **no**, data only |

Notebook 1's output doesn't contain the manifest or the images (it read them from notebook 0), so
notebook 0 has to be attached as well. If notebook 1 ran its own data fallback, its output has the
data and notebook 0 isn't needed. §2 handles both cases.

**Outputs** (`/kaggle/working/`):

```
results/explanations/test_explanations.npz   # all maps + masks, row-aligned with test_predictions.csv
results/explanations/meta.json               # how the maps were made (target, steps, baseline, normalisation)
results/explanations/test_predictions.csv    # copy, so notebook 3 has the row key next to the arrays
results/figures/explanations_curated.png
results/figures/explanations_pred_vs_true_target.png
src/                                         # notebook 1's src/ + src/explain/
```

## 1. GPU check and dependencies

Integrated Gradients with 50 steps runs 50 forward and backward passes per image, about 23k
passes over the whole test set. On a GPU that takes a couple of minutes; on CPU it isn't practical.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Session options -> Accelerator -> GPU, then restart the session.")
DEVICE = torch.device("cuda")
print("device:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
!pip install -q captum

In [ ]:
import hashlib
import json
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import captum
from skimage import measure
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

WORK = Path("/kaggle/working")
EXPL_DIR = WORK / "results" / "explanations"
FIG_DIR = WORK / "results" / "figures"
for d in [EXPL_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("captum", captum.__version__)

## 2. Locate inputs

### 2a. Notebook 1's output (required)

I look for `results/checkpoints/best_model.pt` at any depth under `/kaggle/input/`. If it's not found, the cell prints what *is* mounted, so you can see whether notebook 1 is missing or attached without a checkpoint. If it's missing, the
notebook **stops**. Retraining here would produce a different model from the one whose test
metrics were reported, and every audit number downstream would then describe a model nobody
evaluated. If more than one candidate turns up (say two versions of notebook 1 attached), it
also stops, rather than guess which one you meant.

In [ ]:
def find_nb1_output():
    # any depth: Kaggle's mount layout for notebook outputs has changed more than once
    hits = sorted(Path("/kaggle/input").rglob("best_model.pt"))
    return [h for h in hits if h.parent.name == "checkpoints" and h.parent.parent.name == "results"]


def show_inputs(root=Path("/kaggle/input"), max_depth=4, max_per_dir=8):
    # what is actually mounted, skipping the 3064 .mat files
    def walk(d, depth):
        kids = sorted(d.iterdir())
        for k in kids[:max_per_dir]:
            print("    " * depth + k.name + ("/" if k.is_dir() else ""))
            if k.is_dir() and depth + 1 < max_depth and k.name != "mat":
                walk(k, depth + 1)
        if len(kids) > max_per_dir:
            print("    " * depth + f"... ({len(kids) - max_per_dir} more)")
    print(f"contents of {root}:")
    if root.exists() and any(root.iterdir()):
        walk(root, 1)
    else:
        print("    (empty -- no inputs attached)")


hits = find_nb1_output()
if len(hits) == 0:
    show_inputs()
    raise FileNotFoundError(
        "results/checkpoints/best_model.pt not found anywhere under /kaggle/input (tree printed above).\n"
        "  - Nothing from notebook 1 listed? Add Input -> Your Work -> Notebooks -> 01_baseline_classifier.\n"
        "  - Notebook 1 listed but no results/checkpoints/? Its committed version produced no checkpoint:\n"
        "    check that version's log (did the GPU check fail because the accelerator was off?) and its\n"
        "    Output tab, re-commit notebook 1 with GPU on, then re-attach the new version here.\n"
        "Not retraining here."
    )
if len(hits) > 1:
    raise RuntimeError(f"several notebook-1 checkpoints attached, keep only one: {hits}")

CKPT_PATH = hits[0]
NB1_DIR = CKPT_PATH.parents[2]
for required in ["src/models/classifier.py", "src/data/dataset.py", "src/data/transforms.py",
                 "results/test_predictions.csv", "results/test_metrics.json"]:
    if not (NB1_DIR / required).exists():
        raise FileNotFoundError(f"notebook 1 output at {NB1_DIR} is missing {required} -- re-commit notebook 1")
print("notebook 1 output:", NB1_DIR)

### 2b. Code: notebook 1's `src/`, copied, plus `src/explain/`

Following your rule, the only code tree used is notebook 1's. Notebook 0's `src/` never touches
`sys.path`. I can't just add `/kaggle/input/<nb1>/` to the path, though, because this notebook
has to **add** `src/explain/`, and inputs are read-only. So notebook 1's `src/` is copied into
`/kaggle/working/src`, the new subpackage is written next to it, and only `/kaggle/working` goes
on the path. The `src` that gets imported is notebook 1's, extended, and it's saved with this
notebook's output for notebook 3.

The assertion after the import confirms that `src` really resolves to `/kaggle/working`. If some
other `src` were already importable (from an earlier cell or a stray path entry), it would fail here.

In [ ]:
shutil.copytree(NB1_DIR / "src", WORK / "src", dirs_exist_ok=True)
(WORK / "src" / "explain").mkdir(exist_ok=True)
(WORK / "src" / "explain" / "__init__.py").touch()

sys.path = [p for p in sys.path if not p.startswith("/kaggle/input")]
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

import src
assert Path(src.__file__).resolve().parent == (WORK / "src").resolve(), src.__file__
print("src package:", Path(src.__file__).parent)

### 2c. Data: the manifest whose MD5 matches the checkpoint

Notebook 1 stored the manifest's MD5 in the checkpoint. I pick whichever attached `manifest.csv`
has that exact hash. That guarantees the "test split" here is the one the model was actually
evaluated on, and not a later re-run of notebook 0 with a different split.

In [ ]:
ckpt_raw = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
WANT_MD5 = ckpt_raw["manifest_md5"]

MANIFEST_PATH = DATA_ROOT = None
candidates = sorted(Path("/kaggle/input").glob("**/manifest.csv"))
for m in candidates:
    if hashlib.md5(m.read_bytes()).hexdigest() != WANT_MD5:
        continue
    first = pd.read_csv(m, nrows=1).file_path.iloc[0]
    for root in [m.parent / "data", m.parent.parent / "data"]:
        if (root / first).exists():
            MANIFEST_PATH, DATA_ROOT = m, root
            break
    if MANIFEST_PATH:
        break

if MANIFEST_PATH is None:
    raise FileNotFoundError(
        f"no attached manifest.csv with md5 {WANT_MD5} and its data/ folder. Attach notebook 0's "
        f"committed output (the version notebook 1 trained on). Candidates seen: {candidates}"
    )
print("manifest :", MANIFEST_PATH, "(md5 matches checkpoint)")
print("data root:", DATA_ROOT)

## 3. Load the model

`load_checkpoint` (from notebook 1's `classifier.py`) rebuilds the ResNet18 with
`pretrained=False`, so no ImageNet download is needed, loads the trained weights, and returns the
model in `eval()` mode. Eval mode matters twice here: dropout is off, so a map doesn't change from
one call to the next, and BatchNorm uses its running statistics, so an image's explanation doesn't
depend on whatever else is in the batch.

The config comes from the checkpoint itself, not from a YAML file: it records exactly what this
model was trained with.

In [ ]:
from src.utils.seed import set_seed
from src.models.classifier import load_checkpoint
from src.data.dataset import BrainTumorDataset
from src.data.transforms import get_eval_transforms, IMAGENET_MEAN, IMAGENET_STD

model, ckpt = load_checkpoint(CKPT_PATH, DEVICE)
CFG = ckpt["config"]
CLASS_NAMES = ckpt["class_names"]
SEED = CFG["seed"]
set_seed(SEED)

assert not model.training
print(model.fc)
print("class names:", CLASS_NAMES)
print("checkpoint val metrics:", ckpt["val_metrics"])
CKPT_MD5 = hashlib.md5(CKPT_PATH.read_bytes()).hexdigest()

### Test set, row-aligned with `test_predictions.csv`

Notebook 1 wrote `test_predictions.csv` in the order of `BrainTumorDataset(manifest, split="test")`,
so building the dataset the same way reproduces that order. I assert it file by file anyway.
Every array saved below is indexed by that row order, so an off-by-one here would silently pair
images with the wrong heatmaps.

`return_mask=True`: the eval transform resizes the tumor mask with nearest-neighbour, so it stays
binary and pixel-aligned with the 224×224 input the model sees. That's the mask notebook 3 compares
heatmaps against.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
test_pred_nb1 = pd.read_csv(NB1_DIR / "results" / "test_predictions.csv")
with open(NB1_DIR / "results" / "test_metrics.json") as f:
    test_metrics_nb1 = json.load(f)

IMAGE_SIZE = CFG["image_size"]
test_ds = BrainTumorDataset(manifest, DATA_ROOT, "test", get_eval_transforms(IMAGE_SIZE), return_mask=True)
assert list(test_ds.df.file_path) == list(test_pred_nb1.file_path), "test order differs from test_predictions.csv"
N = len(test_ds)
print(N, "test images,", test_ds.df.patient_id.nunique(), "patients")

### Sanity check: same weights, same predictions

I re-run the whole test set (it's small, a few seconds), not just a few batches, and compare it
image by image with notebook 1's predictions. Agreement should be essentially 100%. It may not be
*exactly* 100%: notebook 1 predicted under fp16 autocast, and this pass runs in fp32 (the gradients
below need fp32), so an image whose top two probabilities are nearly tied can flip. I allow ≥99%
agreement and show any disagreements. Anything worse means the wrong weights or the wrong
preprocessing, and the notebook stops.

The fp32 predictions from this pass are the targets explained below, so both methods explain the
class this exact model in this exact precision chose.

In [ ]:
loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
probs = []
with torch.no_grad():
    for b in loader:
        probs.append(model(b["image"].to(DEVICE)).softmax(1).cpu())
probs = torch.cat(probs).numpy()
pred = probs.argmax(1)
true = test_pred_nb1.true.to_numpy()

agree = (pred == test_pred_nb1.pred.to_numpy()).mean()
acc = (pred == true).mean()
print(f"agreement with notebook 1 predictions: {agree:.4f}")
print(f"test accuracy here {acc:.4f} vs notebook 1 {test_metrics_nb1['accuracy']:.4f}")
flips = np.flatnonzero(pred != test_pred_nb1.pred.to_numpy())
if len(flips):
    display(test_pred_nb1.iloc[flips][["file_path", "true", "pred", "confidence"]].assign(pred_fp32=pred[flips]))
assert agree >= 0.99 and abs(acc - test_metrics_nb1["accuracy"]) < 0.01, "checkpoint does not reproduce notebook 1"

## 4. The two methods, and why audit them

Both produce a heatmap answering "which pixels mattered for this prediction?", but they define
*mattered* very differently.

**Grad-CAM** (Selvaraju et al., 2017) works at the last convolutional block, `layer4`. That block
outputs 512 feature maps at **7×7** for a 224 input. Grad-CAM averages the gradient of the class
score over each map to get one weight per channel ("how much does this feature push the class
score up?"), takes the weighted sum of the maps, applies a ReLU so only positive evidence is kept,
and upsamples to 224×224.
- It's class-discriminative and smooth, and it usually looks convincing.
- Its resolution is fixed at 7×7, so each cell covers a 32×32-pixel patch. Small pituitary tumors
  can occupy less than one cell.
- It measures what the *last block's features* care about, not what the *input pixels* contribute.

**Integrated Gradients** (Sundararajan et al., 2017) works at the **input**. It integrates the
gradient along a straight path from a baseline image (black) to the real image. The attributions
satisfy *completeness*: per-pixel attributions sum to `f(image) − f(baseline)` for the target
logit. Captum reports the remaining error as a *convergence delta*, which I check below.
- Its resolution is pixel-level and it has a clear axiomatic meaning.
- Maps are noisy and speckled, and they depend on the choice of baseline.

**Why audit them?** A heatmap that lands on the tumor *looks* right, but that doesn't prove the
model *used* that region. Methods are known to give plausible-looking maps even on randomised
models (Adebayo et al., 2018). Two methods that disagree can't both be faithful, which is exactly
why notebook 3 scores them instead of just looking at them:
- **faithfulness:** does deleting the top pixels actually change the prediction?
- **stability:** does a 5° rotation move the map?
- **localization:** does the map overlap the tumor mask?

This notebook only has to make sure the maps are computed correctly, and saved without loss of
information.

### 4a. `src/explain/` — shared interface

Both modules expose

```python
explain(model, image_tensor, target=None, **method_kwargs) -> np.ndarray  # (H, W), float32 in [0, 1]
```

- `image_tensor` is one **normalised** image, `(3, H, W)` or `(1, 3, H, W)`: exactly what the
  dataset returns, so notebook 3 can pass dataset items straight through.
- `target=None` means "the model's predicted class"; pass an int to explain another class, e.g.
  the true class of a misclassified image.
- `return_raw=True` additionally returns what was there before normalisation (Grad-CAM's 7×7 map;
  IG's signed per-pixel attribution and convergence delta), for analyses that shouldn't depend
  on my normalisation choices.

`EXPLAINERS` maps the config's method names to the functions, so notebook 3 can loop over
`cfg["explain"]["methods"]` without special-casing either method.

In [ ]:
%%writefile /kaggle/working/src/explain/_common.py
import torch


def as_batch(model, image_tensor):
    # dropout on would make every map a random draw; BN in train mode would mix in batch stats
    assert not model.training, "call model.eval() before explaining"
    x = image_tensor if image_tensor.dim() == 4 else image_tensor.unsqueeze(0)
    assert x.shape[0] == 1, "explain() takes one image"
    device = next(model.parameters()).device
    return x.to(device=device, dtype=torch.float32)


def predicted_class(model, x):
    with torch.no_grad():
        return int(model(x).argmax(1))

### 4b. Grad-CAM

Implementation details that matter:

- **Activations via a forward hook, gradients via `torch.autograd.grad`**, not a backward hook.
  `register_full_backward_hook` on a ResNet block conflicts with its in-place ReLUs ("output is a
  view and is being modified inplace"). Asking autograd directly for d(score)/d(activation) gives
  the same tensor without that conflict, and it doesn't write into any parameter's `.grad`.
- **The score is the pre-softmax logit**, as in the paper. Gradients of the softmax probability
  depend on the other classes and shrink towards zero as confidence saturates near 1.0.
- **Normalised by dividing by the max, not min-max.** After the ReLU the minimum is 0 by
  construction. Min-max would stretch a uniformly weak map to full contrast, so 0 would no longer
  mean "no evidence". If the map is all zeros (no positive evidence anywhere, which happens on
  some misclassifications), it stays all zeros; it isn't divided by ~0.
- Upsampled with bilinear interpolation from 7×7 to the input size, which is the standard choice.
  Notebook 3 should remember that the true resolution is 7×7.

In [ ]:
%%writefile /kaggle/working/src/explain/gradcam.py
import numpy as np
import torch
import torch.nn.functional as F

from src.explain._common import as_batch, predicted_class


def _resolve(model, dotted):
    layer = model
    for part in dotted.split("."):
        layer = getattr(layer, part)
    return layer


def explain(model, image_tensor, target=None, target_layer="layer4", return_raw=False):
    x = as_batch(model, image_tensor)
    if target is None:
        target = predicted_class(model, x)
    layer = _resolve(model, target_layer)

    store = {}
    handle = layer.register_forward_hook(lambda mod, inp, out: store.__setitem__("act", out))
    try:
        with torch.enable_grad():
            # detach so the caller's tensor isn't flagged; requires_grad keeps the graph alive even with frozen weights
            x = x.detach().requires_grad_(True)
            logits = model(x)
            grads, = torch.autograd.grad(logits[0, int(target)], store["act"])
    finally:
        handle.remove()

    act = store["act"].detach()
    weights = grads.mean(dim=(2, 3), keepdim=True)            # (1, C, 1, 1)
    cam = F.relu((weights * act).sum(dim=1, keepdim=True))    # (1, 1, h, w)
    up = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    peak = up.max()
    heat = (up / peak if peak > 0 else torch.zeros_like(up)).cpu().numpy().astype(np.float32)

    if return_raw:
        return heat, {"cam_lowres": cam[0, 0].cpu().numpy(), "target": int(target)}
    return heat

### 4c. Integrated Gradients

- **"Black" baseline in normalised space.** The model sees ImageNet-normalised inputs, so a black
  image is `(0 − mean) / std` per channel, **not** a zero tensor. A zero tensor would be mid-grey
  after un-normalising. Black is a sensible baseline for this data because the MRI background is
  already black after per-image min-max scaling, so the path goes from "empty scan" to "this scan".
- **Channel reduction:** sum the signed attributions over the 3 channels (they're identical copies
  of one grey image, so summing gives the per-pixel total), then take `|·|`. Pixels with strong
  *negative* attribution also matter to the decision. Faithfulness tests that delete
  "important pixels" in notebook 3 should treat them as important.
- **Normalisation: divide by the 99th percentile and clip to [0, 1]**, not divide by the max. IG
  maps are dominated by a handful of extreme pixels. With max-normalisation almost the whole map
  would sit near 0, nothing would reach the 0.5 localization threshold, and notebook 3 would score
  IG as "doesn't localise" for a purely cosmetic reason. Captum's own visualiser clips outliers for
  the same reason. The raw signed map is saved too, so notebook 3 can test whether this choice
  matters.
- **Steps:** `n_steps=50` by default (configurable; `explain.ig_steps` in the config is used if
  present). `internal_batch_size=50` runs the whole path in one batch on a T4. Reduce it if you hit
  out-of-memory errors; the result is identical, just slower.

In [ ]:
%%writefile /kaggle/working/src/explain/integrated_gradients.py
import numpy as np
import torch
from captum.attr import IntegratedGradients

from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD
from src.explain._common import as_batch, predicted_class


def black_baseline(x):
    # black in pixel space == -mean/std in the normalised space the model actually sees
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=x.device).view(1, 3, 1, 1)
    return ((0.0 - mean) / std).expand_as(x).contiguous()


def explain(model, image_tensor, target=None, n_steps=50, internal_batch_size=50,
            clip_percentile=99.0, return_raw=False):
    x = as_batch(model, image_tensor)
    if target is None:
        target = predicted_class(model, x)

    ig = IntegratedGradients(model)
    attr, delta = ig.attribute(x, baselines=black_baseline(x), target=int(target), n_steps=n_steps,
                               internal_batch_size=internal_batch_size, return_convergence_delta=True)
    signed = attr[0].sum(dim=0).detach().float().cpu().numpy()   # (H, W)

    mag = np.abs(signed)
    scale = np.percentile(mag, clip_percentile)
    heat = np.clip(mag / scale, 0.0, 1.0).astype(np.float32) if scale > 0 else np.zeros_like(mag)

    if return_raw:
        return heat, {"signed": signed.astype(np.float32), "delta": float(delta.item()), "target": int(target)}
    return heat

In [ ]:
%%writefile /kaggle/working/src/explain/__init__.py
from src.explain import gradcam, integrated_gradients

# keys match cfg["explain"]["methods"]
EXPLAINERS = {
    "gradcam": gradcam.explain,
    "integrated_gradients": integrated_gradients.explain,
}

### 4d. Quick check on one image

Before running ~460 images, a check on one. Both maps should be `(224, 224)` and in [0, 1]. For
the IG convergence delta, I compare it to the quantity it should match, `f(x) − f(baseline)`.
A relative error of a few percent or less means 50 steps are enough. If it's much larger, raise
`IG_STEPS`.

In [ ]:
import importlib
import src.explain._common, src.explain.gradcam, src.explain.integrated_gradients, src.explain
# re-running the %%writefile cells doesn't update already-imported modules; reload in dependency order
for mod in [src.explain._common, src.explain.gradcam, src.explain.integrated_gradients, src.explain]:
    importlib.reload(mod)
from src.explain import EXPLAINERS, gradcam, integrated_gradients

TARGET_LAYER = CFG["explain"]["target_layer"]
IG_STEPS = CFG["explain"].get("ig_steps", 50)
assert set(CFG["explain"]["methods"]) <= set(EXPLAINERS), CFG["explain"]["methods"]

s = test_ds[0]
t0 = time.time(); gc_map, gc_raw = gradcam.explain(model, s["image"], target_layer=TARGET_LAYER, return_raw=True); t_gc = time.time() - t0
t0 = time.time(); ig_map, ig_raw = integrated_gradients.explain(model, s["image"], n_steps=IG_STEPS, return_raw=True); t_ig = time.time() - t0

with torch.no_grad():
    x = s["image"][None].to(DEVICE)
    logit_gap = (model(x)[0, ig_raw["target"]] - model(integrated_gradients.black_baseline(x))[0, ig_raw["target"]]).item()

print(f"Grad-CAM {gc_map.shape} [{gc_map.min():.2f}, {gc_map.max():.2f}]  lowres {gc_raw['cam_lowres'].shape}  {t_gc*1000:.0f} ms")
print(f"IG       {ig_map.shape} [{ig_map.min():.2f}, {ig_map.max():.2f}]  {t_ig*1000:.0f} ms")
print(f"IG completeness: sum(attr) = {ig_raw['signed'].sum():.4f}, f(x)-f(baseline) = {logit_gap:.4f}, "
      f"delta = {ig_raw['delta']:.4f} ({abs(ig_raw['delta']) / (abs(logit_gap) + 1e-8):.1%} of the gap)")

## 5. Explain the full test set and save for notebook 3

One pass over every test image, both methods, explaining the **fp32 predicted class** from §3.
Notebook 3 asks "did the evidence the model used for its decision lie on the tumor?", so the
decision being explained has to be the one the model actually made. The curated figure in §6
shows what changes when the true class is explained instead.

All arrays are row-aligned with `test_predictions.csv`. Row `i` of every array is row `i` of the
CSV, and the `file_path` array is stored too, so this can be checked after loading. Storage:

| key | shape | dtype | note |
|---|---|---|---|
| `gradcam` | (N, 224, 224) | float16 | normalised map in [0, 1] |
| `gradcam_lowres` | (N, 7, 7) | float32 | pre-upsampling, post-ReLU, unnormalised |
| `ig` | (N, 224, 224) | float16 | \|attr\| / p99, clipped to [0, 1] |
| `ig_signed` | (N, 224, 224) | float32 | raw channel-summed attribution, for renormalising later |
| `ig_delta` | (N,) | float32 | completeness error |
| `mask` | (N, 224, 224) | uint8 | resized tumor mask, {0, 1} |
| `target`, `pred`, `true` | (N,) | int64 | explained class (= fp32 pred), and labels |

float16 is enough for [0, 1] maps (about 3 significant digits). `ig_signed` stays float32, because
per-pixel IG values can be around 1e-5, where float16 loses most of its precision.

In [ ]:
H = W = IMAGE_SIZE
gc_all = np.zeros((N, H, W), np.float16)
gc_low = None
ig_all = np.zeros((N, H, W), np.float16)
ig_signed = np.zeros((N, H, W), np.float32)
ig_delta = np.zeros(N, np.float32)
masks = np.zeros((N, H, W), np.uint8)

t0 = time.time()
for i in tqdm(range(N), desc="explaining test set"):
    s = test_ds[i]
    tgt = int(pred[i])
    g, g_raw = gradcam.explain(model, s["image"], target=tgt, target_layer=TARGET_LAYER, return_raw=True)
    a, a_raw = integrated_gradients.explain(model, s["image"], target=tgt, n_steps=IG_STEPS, return_raw=True)
    if gc_low is None:
        gc_low = np.zeros((N, *g_raw["cam_lowres"].shape), np.float32)
    gc_all[i], gc_low[i] = g, g_raw["cam_lowres"]
    ig_all[i], ig_signed[i], ig_delta[i] = a, a_raw["signed"], a_raw["delta"]
    masks[i] = s["tumor_mask"].numpy()
print(f"{time.time() - t0:.0f}s for {N} images")

In [ ]:
out_npz = EXPL_DIR / "test_explanations.npz"
np.savez_compressed(
    out_npz,
    file_path=np.array(test_ds.df.file_path.tolist()),
    gradcam=gc_all, gradcam_lowres=gc_low,
    ig=ig_all, ig_signed=ig_signed, ig_delta=ig_delta,
    mask=masks, target=pred.astype(np.int64), pred=pred.astype(np.int64), true=true.astype(np.int64),
)
shutil.copy(NB1_DIR / "results" / "test_predictions.csv", EXPL_DIR / "test_predictions.csv")

meta = {
    "n": int(N),
    "row_key": "test_predictions.csv row order (file_path array included)",
    "target": "fp32 predicted class of the loaded checkpoint",
    "agreement_with_nb1_preds": float(agree),
    "image_size": IMAGE_SIZE,
    "class_names": CLASS_NAMES,
    "gradcam": {"target_layer": TARGET_LAYER, "score": "pre-softmax logit", "normalisation": "divide by max (0 if all-zero)",
                "upsampling": "bilinear, align_corners=False", "lowres_shape": list(gc_low.shape[1:])},
    "integrated_gradients": {"n_steps": IG_STEPS, "baseline": "black in pixel space (= -mean/std normalised)",
                             "channel_reduction": "sum over channels, then abs", "normalisation": "divide by 99th percentile, clip [0,1]",
                             "method": "captum default (gausslegendre)"},
    "mask": "tumorMask resized with eval transform (nearest), uint8 {0,1}",
    "checkpoint": str(CKPT_PATH), "checkpoint_md5": CKPT_MD5, "manifest_md5": WANT_MD5,
    "versions": {"torch": torch.__version__, "captum": captum.__version__},
}
with open(EXPL_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

Reload from disk and check everything against `test_predictions.csv`. This tests the saved file,
not the arrays still in memory, since the file is what notebook 3 will read.

In [ ]:
z = np.load(out_npz)
key = pd.read_csv(EXPL_DIR / "test_predictions.csv")
for k in z.files:
    print(f"{k:15s} {str(z[k].shape):18s} {z[k].dtype}")

assert len(key) == N and all(len(z[k]) == N for k in z.files), "row count mismatch"
assert (z["file_path"] == key.file_path.to_numpy()).all(), "row order mismatch"
assert (z["true"] == key.true.to_numpy()).all()
for k in ["gradcam", "ig"]:
    assert np.isfinite(z[k]).all() and z[k].min() >= 0 and z[k].max() <= 1, k
assert set(np.unique(z["mask"])) <= {0, 1}
print(f"\nall {N} rows match test_predictions.csv")
print(f"empty Grad-CAM maps (no positive evidence): {(z['gradcam'].reshape(N, -1).max(1) == 0).sum()}")
print(f"empty masks after resize: {(z['mask'].reshape(N, -1).sum(1) == 0).sum()}")

abs_delta = np.abs(z["ig_delta"])
print(f"IG |delta| (logit units) median {np.median(abs_delta):.4f}, 95th pct {np.percentile(abs_delta, 95):.4f}, max {abs_delta.max():.4f}")
# TODO: if the 95th pct delta is a sizeable fraction of typical logit gaps, rerun with ig_steps=100
!du -sh {out_npz}

## 6. Curated visual check

For each class, up to **2 correctly and 2 incorrectly classified** test images, drawn with
`np.random.default_rng(42)`, so the same images come back on every run and in the committed
version. A class with fewer than 2 errors just shows what it has. The rows are drawn from the
**saved** arrays, so the figure shows exactly what notebook 3 will audit.

Columns: original MRI | Grad-CAM overlay | IG overlay | ground-truth mask contour. A thin white
contour on the overlays makes it easier to see by eye whether a map sits on the tumor.

This is an eyeball check, not evidence; quantifying it is notebook 3's job. Still worth looking for:
- Grad-CAM blobs that cover the tumor plus a lot of surrounding tissue (the 7×7 resolution),
  or that sit on the skull or image border. The second would suggest a shortcut.
- IG speckle concentrated on edges and high-contrast boundaries, which is typical.
- On misclassified images: does either method point somewhere obviously irrelevant?

In [ ]:
mean0, std0 = IMAGENET_MEAN[0], IMAGENET_STD[0]


def to_display(img_tensor):
    # one channel is enough (all three are copies), back to [0, 1]
    return (img_tensor[0].numpy() * std0 + mean0).clip(0, 1)


def draw_contour(ax, mask, **kw):
    for c in measure.find_contours(mask.astype(float), 0.5):
        ax.plot(c[:, 1], c[:, 0], **kw)


rng = np.random.default_rng(SEED)
correct = pred == true
curated = []
for k, cname in enumerate(CLASS_NAMES):
    for is_correct, n_want in [(True, 2), (False, 2)]:
        pool = np.flatnonzero((true == k) & (correct == is_correct))
        picks = np.sort(rng.choice(pool, size=min(n_want, len(pool)), replace=False)) if len(pool) else []
        if len(pool) < n_want:
            print(f"{cname}: only {len(pool)} {'correct' if is_correct else 'misclassified'} test images")
        curated += [int(i) for i in picks]
print("curated rows:", curated)

fig, axes = plt.subplots(len(curated), 4, figsize=(12, 3.1 * len(curated)))
axes = np.atleast_2d(axes)
for r, i in enumerate(curated):
    img, m = to_display(test_ds[i]["image"]), z["mask"][i]
    status = "correct" if correct[i] else "WRONG"
    axes[r, 0].imshow(img, cmap="gray")
    axes[r, 0].set_title(f"true {CLASS_NAMES[true[i]]} | pred {CLASS_NAMES[pred[i]]}\n"
                         f"p={probs[i, pred[i]]:.3f} ({status})", fontsize=9,
                         color="black" if correct[i] else "crimson")
    for c, (name, heat) in enumerate([("Grad-CAM", z["gradcam"][i]), ("Integrated Gradients", z["ig"][i])], start=1):
        axes[r, c].imshow(img, cmap="gray")
        axes[r, c].imshow(heat.astype(np.float32), cmap="jet", alpha=0.45, vmin=0, vmax=1)
        draw_contour(axes[r, c], m, color="white", lw=0.8, ls="--")
        axes[r, c].set_title(name, fontsize=9)
    axes[r, 3].imshow(img, cmap="gray")
    draw_contour(axes[r, 3], m, color="red", lw=1.3)
    axes[r, 3].set_title(f"tumor mask | {Path(key.file_path[i]).name}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
fig.savefig(FIG_DIR / "explanations_curated.png", dpi=130, bbox_inches="tight")
plt.show()

### Explaining the predicted class vs the true class

For misclassified images, a natural question is: *what would the model have needed to look at to
get it right?* Using `target=`, this compares the map for the class the model chose with the map
for the class it should have chosen. When the true-class map does land on the tumor, the model
had some evidence for the right answer and the wrong class outweighed it. That's a different
failure from attending to the wrong region entirely. Notebook 5 goes into this properly; here
it mainly shows that the `target` option works.

In [ ]:
wrong = [i for i in curated if not correct[i]][:6]
if not wrong:
    print("no misclassified images among the curated rows")
else:
    fig, axes = plt.subplots(len(wrong), 5, figsize=(15, 3.1 * len(wrong)))
    axes = np.atleast_2d(axes)
    for r, i in enumerate(wrong):
        s = test_ds[i]
        img, m = to_display(s["image"]), z["mask"][i]
        maps = [
            (f"Grad-CAM -> pred ({CLASS_NAMES[pred[i]]})", z["gradcam"][i]),
            (f"Grad-CAM -> true ({CLASS_NAMES[true[i]]})", gradcam.explain(model, s["image"], target=int(true[i]), target_layer=TARGET_LAYER)),
            (f"IG -> pred ({CLASS_NAMES[pred[i]]})", z["ig"][i]),
            (f"IG -> true ({CLASS_NAMES[true[i]]})", integrated_gradients.explain(model, s["image"], target=int(true[i]), n_steps=IG_STEPS)),
        ]
        axes[r, 0].imshow(img, cmap="gray")
        draw_contour(axes[r, 0], m, color="red", lw=1.3)
        axes[r, 0].set_title(f"{Path(key.file_path[i]).name}\np(pred)={probs[i, pred[i]]:.2f} p(true)={probs[i, true[i]]:.2f}", fontsize=9)
        for c, (title, heat) in enumerate(maps, start=1):
            axes[r, c].imshow(img, cmap="gray")
            axes[r, c].imshow(np.asarray(heat, np.float32), cmap="jet", alpha=0.45, vmin=0, vmax=1)
            draw_contour(axes[r, c], m, color="white", lw=0.8, ls="--")
            axes[r, c].set_title(title, fontsize=8)
    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "explanations_pred_vs_true_target.png", dpi=130, bbox_inches="tight")
    plt.show()

In [ ]:
!ls -la /kaggle/working/results/explanations /kaggle/working/results/figures /kaggle/working/src/explain

## Closing checklist

**Correctness**
- [ ] Prediction agreement with notebook 1 ≥ 99% and test accuracy within 1 point (the cell in §3
  asserts this). If it failed, the checkpoint, manifest or preprocessing doesn't match. Fix that
  before anything else; every explanation depends on it.
- [ ] Single-image check: both maps `(224, 224)` in [0, 1], and IG's delta a small fraction of
  `f(x) − f(baseline)`.
- [ ] `test_explanations.npz` reloads with **N rows matching `test_predictions.csv`**, in the same
  order (asserted).
- [ ] Number of empty Grad-CAM maps is small. A large count would mean the class logit's gradient
  is mostly negative at `layer4`, which is worth understanding before notebook 3.
- [ ] Curated figure: Grad-CAM maps are blobs at roughly 7×7 resolution, and IG maps are
  pixel-level speckle. If Grad-CAM looks pixel-sharp, it isn't hooked to `layer4`.

**Save & Run All**
1. **Save Version → Save & Run All (Commit) → Save**, with the GPU accelerator on and both inputs
   (notebook 0 and notebook 1 outputs) still attached. The commit runs in a fresh session and
   needs both.
2. After it finishes, open the version's **Output** tab and check for
   `results/explanations/test_explanations.npz`, `meta.json`, `test_predictions.csv`, both figures,
   and `src/explain/`.
3. **For notebook 3**, attach: notebook 0 (data), notebook 1 (checkpoint, which deletion/insertion
   needs in order to re-run the model on perturbed images), and this notebook (explanations +
   `src/`). Put only **this** notebook's `src/` on `sys.path`; it's notebook 1's `src/` plus
   `src/explain/`.